In [ ]:
%load_ext autoreload
%autoreload 

In [ ]:
import pandas as pd
import numpy as np
from os import path, makedirs
from datetime import datetime

# local imports
import sys
sys.path.append('../../../')
from pyanalib.split_df_helpers import *
from analysis_village.cc1pi.systematics.final_variable_configs import VariableConfig
from analysis_village.cc1pi.systematics.utils import *
from analysis_village.cc1pi.systematics.constants import *
from pyanalib.covariance import *
from analysis_village.cc1pi.DataFrameUtils.DFLoading import *

from makedf.mcstat import get_MCstat_unc

# turn off PerformanceWarning 
# triggered by mismatched column levels
import warnings
warnings.filterwarnings("ignore", category=pd.errors.PerformanceWarning)

In [ ]:
save_result = True
save_fig = save_result

save_fig_base_dir = "/exp/sbnd/data/users/lpelegri/syst/"

today_str = datetime.now().strftime("%Y%m%d")
save_fig_dir = path.join(save_fig_base_dir, "systematics-other-{}".format(today_str))

if save_fig:
    if not path.exists(save_fig_dir):
        makedirs(save_fig_dir)
    print("saving plots in ", save_fig_dir)

# Load df

In [ ]:
pot_weight_col = ('slc', 'wgt', '', '', '', '')

#Load CV dataframe
keys2load = ["cc1pi", "hdr", "histpotdf", "nudf"] ## keys from the configuration file
mc_bnb_df = load_df("/exp/sbnd/data/users/lpelegri/cafpyana_data/cc1pi_1e20_training.df", keys2load, 100)
mc_evt_df = mc_bnb_df['cc1pi']
mc_nu_df = mc_bnb_df['nudf']
mc_hdr_df = mc_bnb_df['hdr']

#Add weight column
data_tot_pot = 5.947e+18
mc_tot_pot = mc_hdr_df['pot'].sum()
print("mc_tot_pot: %.3e" %(mc_tot_pot))
mc_pot_scale = data_tot_pot / mc_tot_pot
print("mc_pot_scale: %.3e" %(mc_pot_scale))
mc_evt_df[pot_weight_col] = mc_pot_scale * np.ones(len(mc_evt_df))

#Do truth matchign
mc_evt_df = perform_truth_matching(mc_evt_df, mc_nu_df)
mc_nu_df[pot_weight_col] = mc_pot_scale * np.ones(len(mc_nu_df))

# Perform selection

In [ ]:
mc_obvious_cosmic_mask = mc_evt_df.slc.cut.obvious_cosmic
mc_t0_mask = mc_evt_df.slc.cut.t0
mc_is_inside_FV_mask = mc_evt_df.slc.cut.inside_FV
mc_nu_score_mask = mc_evt_df.slc.cut.nu_score
mc_track_mask = mc_evt_df.slc.cut.track
mc_shower_mask = mc_evt_df.slc.cut.shower 
mc_chi2_mask = mc_evt_df.slc.cut.MIP_candidates 
mc_angle_mask = mc_evt_df.slc.cut.angle 
mc_proton_BDT_mask = mc_evt_df.slc.cut.proton_BDT
mc_containment_mask = mc_evt_df.slc.cut.containment 
mc_michel_mask = mc_evt_df.slc.cut.michel 
mc_extra_pion_mask = mc_evt_df.slc.cut.extra_pion 
mc_energy_mask = mc_evt_df.slc.cut.energy

# 1. Define the order of cuts
mc_cut_sequence = [
    ("cosmic", mc_obvious_cosmic_mask),
    ("t0", mc_t0_mask),
    ("FV", mc_is_inside_FV_mask),
    ("nu_score", mc_nu_score_mask),
    ("track", mc_track_mask),
    ("chi2", mc_chi2_mask),
    ("shower", mc_shower_mask),
    ("angle", mc_angle_mask),
    ("proton_BDT", mc_proton_BDT_mask),
    ("containment", mc_containment_mask),
    ("michel", mc_michel_mask),
    ("extra_pion", mc_extra_pion_mask),
    ("energy", mc_energy_mask)
]

# 2. Build the cumulative masks
mc_cumulative_mak = None

for name, mask in mc_cut_sequence:
    if mc_cumulative_mak is None:
        mc_cumulative_mak = mask
    else:
        mc_cumulative_mak = mc_cumulative_mak & mask

In [ ]:
print(mc_evt_df.truth.nu_categ.value_counts())

In [ ]:
mc_evt_df = mc_evt_df[mc_cumulative_mak]

In [ ]:
#make it a slc df
mc_evt_df = (
        mc_evt_df
        .groupby(['__ntuple', 'entry', 'rec.slc..index'])
        .first()
    )
mc_evt_df = mc_evt_df.sort_index()


In [ ]:
print("==== breakdown of selected events ====")
print(mc_evt_df.truth.nu_categ.value_counts())
#print(mc_evt_df.genie_categ.value_counts())

# MC Stat

In [ ]:
import hashlib
def get_MCstat_unc(evt_df, hdr_df, n_universes=100):
    # Create a unique seed based on event metadata
    # Using a hash function that's deterministic
    meta_seeds = []
    for i in tqdm(range(len(evt_df))):
        this_hdr_df = hdr_df.loc[evt_df.reset_index(level=[2]).index[i]]
        runno = this_hdr_df.run
        subrunno = this_hdr_df.subrun
        evtno = this_hdr_df.evt
        slcid = mc_evt_df.loc[mc_evt_df.index[i]].slc.self
        seed_string = f"run_{runno}_subrun_{subrunno}_evt_{evtno}_slcid_{slcid}"
        #unique_seed = hash(f"run_{runno}_subrun_{subrunno}_evt_{evtno}_slcid_{slcid}") % (2**32)  # Ensure it's a 32-bit integer
        unique_seed = int(
            hashlib.sha256(seed_string.encode()).hexdigest(),
            16
        ) % (2**32)
        if unique_seed in meta_seeds:
            print("duplicate seed found", unique_seed)
            break
        meta_seeds.append(unique_seed)

    # make sure the seeds are unique!
    assert len(meta_seeds) == len(set(meta_seeds))

    # generate universes
    MCstat_univ_events = np.zeros((n_universes, len(evt_df)))
    poisson_mean = 1.0

    # get Poisson weights and save to "MCstat.univ_"
    # dummy df to hold the weights -- iterative inserting causes PerformanceWarning
    mcstat_univ_cols = pd.MultiIndex.from_product(
        [["truth"], ["MCstat"], [f"univ_{i}" for i in range(n_universes)],[""],[""],[""]],
    )
    mcstat_univ_wgt = pd.DataFrame(
        1.0,
        index=evt_df.index,
        columns=mcstat_univ_cols,
    )

    for uidx in range(n_universes):
        universe_string = f"universe_{uidx}"
        universe_seed = int(
            hashlib.sha256(universe_string.encode()).hexdigest(),
            16
        ) % (2**32)
            
        poisson_weights = []
        for sidx, meta_seed in enumerate(meta_seeds):
            # Combine universe seed with event seed for unique randomness -- per event, per universe
            combined_seed = (universe_seed + meta_seed) % (2**32)
            np.random.seed(combined_seed)
            
            poisson_val = np.random.poisson(poisson_mean)
            poisson_weights.append(poisson_val)
            
        mcstat_univ_wgt[("truth","MCstat", "univ_{}".format(uidx),'','','')] = np.array(poisson_weights)
        MCstat_univ_events[uidx, :] = np.array(poisson_weights)

    evt_df = evt_df.join(mcstat_univ_wgt)
    return evt_df, MCstat_univ_events

In [ ]:
mc_evt_df, MCstat_univ_events = get_MCstat_unc(mc_evt_df, mc_hdr_df, n_universes=100)

In [ ]:
import numpy as np
import inspect

class VariableConfig2D:

    def __init__(self,
                 var_save_name,
                 var_plot_name,
                 var_unit,
                 bins_x,
                 bins_y,
                 var_evt_reco_col_x,
                 var_evt_truth_col_x,
                 var_nu_col_x,
                 var_evt_reco_col_y,
                 var_evt_truth_col_y,
                 var_nu_col_y,
                 xsec_label):

        self.var_save_name = var_save_name
        self.var_plot_name = var_plot_name
        self.var_unit = var_unit
        self.xsec_label = xsec_label

        self.bins_x = bins_x
        self.bins_y = bins_y

        self.n_bins_x = len(bins_x) - 1
        self.n_bins_y = len(bins_y) - 1
        self.n_bins_total = self.n_bins_x * self.n_bins_y

        self.bin_centers_x = (bins_x[:-1] + bins_x[1:]) / 2
        self.bin_centers_y = (bins_y[:-1] + bins_y[1:]) / 2

        self.var_evt_reco_col_x = var_evt_reco_col_x
        self.var_evt_truth_col_x = var_evt_truth_col_x
        self.var_nu_col_x = var_nu_col_x

        self.var_evt_reco_col_y = var_evt_reco_col_y
        self.var_evt_truth_col_y = var_evt_truth_col_y
        self.var_nu_col_y = var_nu_col_y

    # -------------------------------------------------------
    # flatten 2D bin index -> 1D index
    # -------------------------------------------------------
    def flatten_index(self, x_bin, y_bin):
        return y_bin * self.n_bins_x + x_bin

    # -------------------------------------------------------
    # convert arrays of x,y values to flattened bin index
    # -------------------------------------------------------
    def digitize_to_flat_index(self, x_vals, y_vals):

        x_bin = np.digitize(x_vals, self.bins_x) - 1
        y_bin = np.digitize(y_vals, self.bins_y) - 1
    
        # clip underflow/overflow into edge bins
        x_bin = np.clip(x_bin, 0, self.n_bins_x - 1)
        y_bin = np.clip(y_bin, 0, self.n_bins_y - 1)
    
        flat_index = self.flatten_index(x_bin, y_bin)
    
        return flat_index

    # -------------------------------------------------------
    # histogram builder
    # -------------------------------------------------------
    def histogram1d(self, x_vals, y_vals, weights=None):
        flat_index = self.digitize_to_flat_index(x_vals, y_vals)
    
        hist = np.bincount(
            flat_index,
            weights=weights,
            minlength=self.n_bins_total
        )
    
        bins = np.arange(self.n_bins_total + 1)
    
        return hist, bins

    # -------------------------------------------------------
    # example configuration
    # -------------------------------------------------------
    @classmethod
    def muon_momentum_muon_direction(cls):

        return cls(
            var_save_name="muon_p_muon_cos_theta",
            var_plot_name=r"$P_\mu$",
            var_unit="[GeV/c]",

            bins_x=np.array([0.1,0.25,0.4,0.6,0.85,3]),
            bins_y=np.array([-1.,0.55,0.75,0.9,1.]),
            #bins_x=np.linspace(0, 3, 11),  # 5 edges = 4 bins
            #bins_y=np.array([-1.,0,0.35,0.53,0.66,0.76,0.84,0.9,0.95,1.]),
            
            var_evt_reco_col_x=('slc','measure_var','reco_p_mu',''),
            var_evt_truth_col_x=('truth','true_var','true_p_mu',''),
            var_nu_col_x=('truth','true_var','true_p_mu',''),

            var_evt_reco_col_y=('slc','measure_var','reco_cos_theta_mu',''),
            var_evt_truth_col_y=('truth','true_var','true_cos_theta_mu',''),
            var_nu_col_y=('truth','true_var','true_cos_theta_mu',''),

            xsec_label=r"$\frac{d\sigma}{dP_\mu}$ ($\mathrm{cm^2}$ [$\mathrm{GeV/c}$])"
        )

    def get_bin_labels(self):

        labels = []
    
        for j in range(self.n_bins_y):
            for i in range(self.n_bins_x):
    
                x0 = self.bins_x[i]
                x1 = self.bins_x[i+1]
                y0 = self.bins_y[j]
                y1 = self.bins_y[j+1]
    
                labels.append(
                    f"Pμ [{x0:.2f},{x1:.2f}], cosθ [{y0:.2f},{y1:.2f}]"
                )
    
        return labels

In [ ]:
import os
import matplotlib.pyplot as plt
import numpy as np
import matplotlib.patheffects as path_effects

file_dir = "/exp/sbnd/data/users/lpelegri/syst2D/frac_cov_matrices"
os.makedirs(file_dir, exist_ok=True) 
show_plots = True

var_config = VariableConfig2D.muon_momentum_muon_direction()
var_configs = [VariableConfig2D.muon_momentum_muon_direction()]
syst_name = "MCstat"

x = mc_evt_df[var_config.var_evt_reco_col_x].values
y = mc_evt_df[var_config.var_evt_reco_col_y].values
w = mc_evt_df[('slc','wgt','','','','')].values

# --- 1. Flattened 1D Histogram (Original Logic) ---
hist_1d, bins_1d = var_config.histogram1d(x, y, w)
bin_centers_1d = (bins_1d[:-1] + bins_1d[1:]) / 2

plt.figure(figsize=(12, 6))
plt.bar(bin_centers_1d, hist_1d, width=0.8, align="center", color='tab:blue', alpha=0.7)

labels = var_config.get_bin_labels()
plt.xticks(bin_centers_1d, labels, rotation=90, fontsize=8)
plt.xlabel("2D Bin (Pμ, cosθ)")
plt.ylabel("Events")
plt.title(f"{var_config.var_save_name} - Flattened")
plt.tight_layout()
plt.show()

# --- 2. New 2D Histogram (X vs Y) ---
# We use the actual physical bin edges from var_config
hist_2d, xedges, yedges = np.histogram2d(x, y, bins=(var_config.bins_x, var_config.bins_y), weights=w)

plt.figure(figsize=(10, 8))
# Use pcolormesh for uneven bin widths (typical in neutrino physics)
X, Y = np.meshgrid(xedges, yedges)
im = plt.pcolormesh(X, Y, hist_2d.T, cmap='viridis', shading='auto')

plt.colorbar(im, label="Events")
plt.xlabel(f"{var_config.var_plot_name} {var_config.var_unit}")
plt.ylabel(r"$\cos\theta_\mu$")
plt.title(f"2D Event Rate: {var_config.var_save_name}")

# Optional: Add text annotations for the event counts in each 2D cell
for i in range(len(xedges)-1):
    for j in range(len(yedges)-1):
        # Only plot text if there are actually events to keep it clean
        if hist_2d[i, j] > 0:
            plt.text(
                var_config.bin_centers_x[i], 
                var_config.bin_centers_y[j], 
                f'{hist_2d[i,j]:.1f}', 
                color='white', 
                ha='center', 
                va='center', 
                fontsize=9,
                # Use the imported module here:
                path_effects=[path_effects.withStroke(linewidth=2, foreground='black')]
            )

plt.tight_layout()
plt.show()

In [ ]:


nudf_signal = mc_nu_df[mc_nu_df.truth.nu_categ == "CC1pi"]
nevts_signal_truth, _ = var_config.histogram1d(
            nudf_signal[var_config.var_nu_col_x].values,
            nudf_signal[var_config.var_nu_col_y].values,
            weights=nudf_signal[('slc','wgt','','','','')].values
        )

evtdf_signal = mc_evt_df[mc_evt_df.truth.nu_categ == "CC1pi"]
nevts_signal_sel_reco, bins_flat = var_config.histogram1d(
            evtdf_signal[var_config.var_evt_reco_col_x].values,
            evtdf_signal[var_config.var_evt_reco_col_y].values,
            weights=evtdf_signal[('slc','wgt','','','','')].values
        )

nevts_signal_sel_truth, _ = var_config.histogram1d(
            evtdf_signal[var_config.var_evt_truth_col_x].values,
            evtdf_signal[var_config.var_evt_truth_col_y].values,
            weights=evtdf_signal[('slc','wgt','','','','')].values
        )


# 2. Plotting
fig, ax = plt.subplots(figsize=(8, 5))



# 1. Total Signal (Generated/Truth Level)
ax.stairs(nevts_signal_truth, bins_flat, 
           label="Total Signal (Truth)", color='black', linewidth=2)

# 2. Selected Signal (Plotted at Reco coordinates)
ax.stairs(nevts_signal_sel_reco, bins_flat, 
           label="Selected Signal (Reco)", color='tab:blue', linewidth=2)

# 3. Selected Signal (Plotted at Truth coordinates - for checking migration)
ax.stairs(nevts_signal_sel_truth, bins_flat, 
           label="Selected Signal (Truth)", color='tab:red', linewidth=2)

# 1. Get the total number of bins from your config
n_bins_total = var_config.n_bins_total  # This should be 225 based on your error

# 2. Define the positions for the center of every bin
# If your x-axis is just the "index" of the bin:
bin_centers = np.arange(n_bins_total) + 0.5
bin_edges = np.arange(n_bins_total + 1)

# 3. Apply the logic to the axes
ax.set_xticks(bin_edges)               # Major ticks on the lines
ax.set_xticks(bin_centers, minor=True)  # Minor ticks in the middle

# Now this will work because len(bin_centers) == len(labels) == 225
ax.set_xticklabels(labels, minor=True, rotation=90, ha='center', fontsize=7)
ax.set_xticklabels([], minor=False)

# Formatting
ax.legend(frameon=False)
ax.set_ylabel("Events")
ax.set_xlabel("Bins") # Or your preferred x-axis label
ax.set_xlim(bins_flat[0], bins_flat[-1])

if save_fig:
    plt.savefig("{}/{}-sel_event_rates.pdf".format(save_fig_dir, var_config.var_save_name), bbox_inches='tight')
plt.show()

In [ ]:
def get_text_color(value, im):
    """
    Determines if text should be black or white based on the 
    background color assigned by the colormap.
    """
    # Get the colormap and normalization from the actual plot object
    cmap = im.get_cmap()
    norm = im.norm
    
    # Get RGBA color for this specific value
    rgba = cmap(norm(value))
    
    # Standard formula for Relative Luminance (perceived brightness)
    # rgba[0,1,2] are R, G, B channels
    luminance = 0.299 * rgba[0] + 0.587 * rgba[1] + 0.114 * rgba[2]
    
    # If the background is bright (>0.5), use black text. Otherwise, white.
    return "black" if luminance > 0.5 else "white"

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def plot_heatmap(matrix, 
                 bins,
                 plot_labels=["", "", ""],
                 approval="internal",
                 verbose=False,
                 plot=True,
                 save_fig=False, 
                 save_name=None,
                 tick_labels=None):
               
    # 1. Setup dimensions and coordinates
    nbins = len(bins)
    # Ensure matrix matches bin dimensions
    assert nbins-1 == matrix.shape[0] == matrix.shape[1]
    
    # Define the display grid (0 to N-1)
    unif_bin = np.linspace(0., float(nbins - 1), nbins)
    bin_centers = (unif_bin[:-1] + unif_bin[1:]) / 2
    extent = [unif_bin[0], unif_bin[-1], unif_bin[0], unif_bin[-1]]

    # 2. Initialize Plot
    fig, ax = plt.subplots(figsize=(10, 10))
    
    # Use 'origin="lower"' so [0,0] is at the bottom left (standard for HEP matrices)
    im = ax.imshow(matrix, extent=extent, origin="lower", cmap='viridis')

    # 3. CONFIGURE TICKS AND LABELS
    # Set Major Ticks on the EDGES
    ax.set_xticks(unif_bin)
    ax.set_yticks(unif_bin)
    
    # Set Minor Ticks on the CENTERS (for the labels)
    ax.set_xticks(bin_centers, minor=True)
    ax.set_yticks(bin_centers, minor=True)

    if tick_labels is not None:
        # Place the text labels on the minor ticks (the centers)
        ax.set_xticklabels(tick_labels, minor=True, rotation=90, ha='center')
        ax.set_yticklabels(tick_labels, minor=True, rotation=0, va='center')
        
        # Remove labels from the major ticks (the edges) so they don't overlap
        ax.set_xticklabels([], minor=False)
        ax.set_yticklabels([], minor=False)
    else:
        # Default behavior: use range labels like "[0.1 - 0.2]"
        x_labels = bin_range_labels(np.array(bins))
        ax.set_xticklabels(x_labels, minor=True, rotation=45, ha="right")
        ax.set_yticklabels(x_labels, minor=True)
        ax.set_xticklabels([], minor=False)
        ax.set_yticklabels([], minor=False)

    # 4. Add Bin Boundary Grid
    # We draw the grid on the major ticks (the edges)
    ax.grid(which='major', color='white', linestyle='-', linewidth=0.5, alpha=0.3)

  # 5. Cell Annotations (Text values with Overlap Prevention)
    # --------------------------------------------------------
    # Determine dynamic font size based on number of bins
    # If nbins > 20, font size shrinks; if > 40, we might hide text
    base_fs = 12
    grid_scale = 1.0 if nbins < 15 else (15 / nbins)
    if nbins < 45: 
        for i in range(nbins-1):      # rows (y)
            for j in range(nbins-1):  # columns (x)
                value = matrix[i, j]
                
                if not np.isnan(value):
                    label = f"{value:.2f}"
                    
                    # Calculate string-based scaling
                    # We treat 4 characters (e.g., '0.21') as the "standard" width
                    # Anything longer gets scaled down proportionally
                    char_len = len(label)
                    string_scale = 4.0 / char_len if char_len > 4 else 1.0
                    
                    # Final font size combines grid density and string length
                    final_fs = base_fs * grid_scale * string_scale
                    
                    ax.text(
                        j + 0.5, i + 0.5,
                        label,
                        ha="center", va="center",   
                        color=get_text_color(value, im),
                        fontsize=final_fs,
                        clip_on=True 
                    )
                    
    elif verbose:
        print(f"Matrix size ({nbins}x{nbins}) is too large for legible text annotations.")
    # 6. Formatting and Labels
    cb = fig.colorbar(im, ax=ax, shrink=0.7)
    cb.set_label(plot_labels[2], fontsize=15)
    
    ax.set_xlabel(plot_labels[0], fontsize=20)
    ax.set_ylabel(plot_labels[1], fontsize=20)

    # Statistical info if verbose
    if verbose:
        n_diag = np.sum(np.diag(matrix))
        total_sum = np.sum(matrix)
        diagonal_ratio = n_diag / total_sum if total_sum != 0 else 0
        print(f"Diagonal ratio: {diagonal_ratio:.2f}")

    # Add approval text (Assumes this helper exists)
    add_approval_text(approval, 0.95, 1.05, "right")

    # 7. Save and Show
    if save_fig:
        # Assumes fig_ext and dpi are defined globally or use defaults
        extension = locals().get('fig_ext', '.pdf')
        res = locals().get('dpi', 300)
        plt.savefig(f"{save_name}{extension}", bbox_inches='tight', dpi=res)

    if plot:
        plt.show()
    else:
        plt.close(fig)

In [ ]:
save_fig_name = "{}/{}-reco_vs_true".format(save_fig_dir, var_config.var_save_name)

true_bins_per_event = var_config.digitize_to_flat_index(
    evtdf_signal[var_config.var_evt_truth_col_x].values,
    evtdf_signal[var_config.var_evt_truth_col_y].values
)

reco_bins_per_event = var_config.digitize_to_flat_index(
    evtdf_signal[var_config.var_evt_reco_col_x].values,
    evtdf_signal[var_config.var_evt_reco_col_y].values
)

# 2. Extract weights (one weight per event)
weights = evtdf_signal[('slc','wgt','','','','')].values

# 3. Build the 2D Histogram (Migration Matrix)
# x-axis: True Bin Index, y-axis: Reco Bin Index
reco_vs_true, xedges, yedges = np.histogram2d(
    true_bins_per_event, 
    reco_bins_per_event, 
    bins=(bins_flat, bins_flat), # bins_flat are the bin edges for the 1D indices
    weights=weights
)


plot_heatmap(reco_vs_true, 
                     bins_flat, 
                     plot_labels=["Bins","Bins", "reco vs true"], tick_labels = labels)

# --- Efficiency ---
nevts_signal_sel_reco, bins_flat = var_config.histogram1d(
            evtdf_signal[var_config.var_evt_reco_col_x].values,
            evtdf_signal[var_config.var_evt_reco_col_y].values,
            weights=evtdf_signal[('slc','wgt','','','','')].values
        )
eff = np.divide(
    nevts_signal_sel_truth, 
    nevts_signal_truth, 
    out=np.zeros_like(nevts_signal_sel_truth), 
    where=nevts_signal_truth != 0
)

# 2. Plotting
fig, ax = plt.subplots(figsize=(8, 5))

# 1. Plot as a step histogram
ax.stairs(eff, bins_flat, color='tab:green', linewidth=2, label="Selection Efficiency")

# 2. Configure the Ticks
# Set Major ticks at the bin EDGES
ax.set_xticks(bins_flat)

# Set Minor ticks at the bin CENTERS (for the labels)
ax.set_xticks(bin_centers, minor=True)

# 3. Apply Labels to Centers
# We use tick_labels for the minor ticks and empty strings for major ticks
ax.set_xticklabels(labels, minor=True, rotation=90, ha='center')
ax.set_xticklabels([], minor=False) 

# Formatting
ax.set_ylim(0, 1.1)
ax.set_ylabel("Efficiency")
ax.set_xlabel("Bins")
ax.grid(axis='y', linestyle='--', alpha=0.7)
ax.set_title(f"Efficiency: {var_config.var_save_name}")
ax.legend()

plt.show()

labels = var_config.get_bin_labels()
response = get_response_matrix(reco_vs_true, eff)
plot_heatmap(response, 
                     bins_flat, 
                     plot_labels=["Bins","Bins", "Response"], tick_labels = labels)


'''
reco_vs_true = get_smear_matrix(var_signal_sel_truth, var_signal_sel_reco, bins_2d, var_labels=var_config.var_labels,
                                save_fig=save_fig, save_fig_name=save_fig_name, weights = weight_signal)
eff = get_eff(reco_vs_true, nevts_signal_truth)
print("eff")
print(eff)

save_fig_name = "{}/{}-response_matrix".format(save_fig_dir, var_config.var_save_name)
Response = get_response_matrix(reco_vs_true, eff, var_config.bins, var_labels=var_config.var_labels,
                               save_fig=save_fig, save_fig_name=save_fig_name)
'''

In [ ]:
def get_univ_rates_2d(cov_type="rate", 
                      evtdf=None, 
                      nudf=None, 
                      var_config=None, 
                      syst_name="", 
                      n_univ=100, 
                      bkgd_subtract=True):
    
    # 1. Setup Scaling and Initial Safety Checks
    if cov_type == "xsec" and nudf is None:
        raise ValueError("nudf (Truth DataFrame) must be provided for cov_type='xsec' to calculate efficiency!")

    if cov_type == "xsec":
        print(f"Getting {n_univ} universes for {syst_name} (xsec)")
        scale_factor = XSEC_UNIT
    else:
        print(f"Getting {n_univ} universes for {syst_name} (rate)")
        scale_factor = 1.0

    # 2. Extract Data and Filter Signal
    # Identify Signal Events in Reco (evtdf)
    evtdf_signal = evtdf[evtdf.truth.nu_categ == "CC1pi"]
    slc_wgt_signal = evtdf_signal[('slc','wgt','','','','')].values

    # Identify Signal Events in Truth (nudf) - FIXED POSITION
    nudf_signal = None
    if nudf is not None:
        nudf_signal = nudf[nudf.truth.nu_categ == "CC1pi"]

    # Build CV Signal Histograms
    nevts_sel_reco_cv, bins_flat = var_config.histogram1d(
        evtdf_signal[var_config.var_evt_reco_col_x].values,
        evtdf_signal[var_config.var_evt_reco_col_y].values,
        weights=slc_wgt_signal
    )
    
    # Truth-level CV for efficiency
    if nudf_signal is not None:
        nevts_allmc_cv, _ = var_config.histogram1d(
            nudf_signal[var_config.var_nu_col_x].values,
            nudf_signal[var_config.var_nu_col_y].values,
            weights=nudf_signal[('slc','wgt','','','','')].values
        )

    # 3. Universe Loop
    univ_events = []
    for uidx in range(n_univ):
        syst_column = ("truth", syst_name, f"univ_{uidx}", "", "", "")
        univ_wgt_signal = np.clip(evtdf_signal[syst_column].fillna(1.0).values, 0, 30)
        
        if cov_type == "xsec":
            # --- Smearing Matrix R_ij ---
            flat_reco_indices = var_config.digitize_to_flat_index(
                evtdf_signal[var_config.var_evt_reco_col_x].values,
                evtdf_signal[var_config.var_evt_reco_col_y].values
            )
            flat_truth_indices = var_config.digitize_to_flat_index(
                evtdf_signal[var_config.var_evt_truth_col_x].values,
                evtdf_signal[var_config.var_evt_truth_col_y].values
            )
            
            reco_vs_true, _, _ = np.histogram2d(
                flat_truth_indices, 
                flat_reco_indices, 
                bins=(bins_flat, bins_flat),
                weights=slc_wgt_signal * univ_wgt_signal
            )

            # --- Efficiency (using nudf_signal safely) ---
            univ_wgt_allmc = nudf_signal[syst_column].fillna(1.0).values
            signal_allmc_univ, _ = var_config.histogram1d(
                nudf_signal[var_config.var_nu_col_x].values,
                nudf_signal[var_config.var_nu_col_y].values,
                weights=nudf_signal[('slc','wgt','','','','')].values * univ_wgt_allmc
            )
            signal_sel_univ, _ = var_config.histogram1d(
                evtdf_signal[var_config.var_evt_truth_col_x].values,
                evtdf_signal[var_config.var_evt_truth_col_y].values,
                weights=slc_wgt_signal * univ_wgt_signal
            )
            
            # Use small epsilon to avoid divide-by-zero
            eff = signal_sel_univ / (signal_allmc_univ + 1e-10)
            response_univ = get_response_matrix(reco_vs_true, eff)
            signal_univ = response_univ @ nevts_allmc_cv 

        elif cov_type == "rate":            
            signal_univ, _ = var_config.histogram1d(
                evtdf_signal[var_config.var_evt_reco_col_x].values,
                evtdf_signal[var_config.var_evt_reco_col_y].values,
                weights=slc_wgt_signal * univ_wgt_signal
            )

        # --- Background Subtraction ---
        for mode in topology_list[1:]:
            this_evtdf = evtdf[evtdf.truth.nu_categ == mode]
            if this_evtdf.empty: continue
                
            bkg_wgt = this_evtdf[('slc','wgt','','','','')].values
            univ_wgt_bkg = np.clip(this_evtdf[syst_column].fillna(1.0).values, 0, 30)
            
            bkg_cv, _ = var_config.histogram1d(
                this_evtdf[var_config.var_evt_reco_col_x].values,
                this_evtdf[var_config.var_evt_reco_col_y].values,
                weights=bkg_wgt
            )
            bkg_univ, _ = var_config.histogram1d(
                this_evtdf[var_config.var_evt_reco_col_x].values,
                this_evtdf[var_config.var_evt_reco_col_y].values,
                weights=bkg_wgt * univ_wgt_bkg
            )
            
            if bkgd_subtract:
                signal_univ += (bkg_univ - bkg_cv)
            else:
                signal_univ += bkg_univ

        univ_events.append(signal_univ * scale_factor)

    # 4. Final Calculations
    univ_events = np.array(univ_events)
    
    if bkgd_subtract:
        cv_events = nevts_sel_reco_cv * scale_factor
    else:
        total_reco_cv, _ = var_config.histogram1d(
            evtdf[var_config.var_evt_reco_col_x].values,
            evtdf[var_config.var_evt_reco_col_y].values,
            weights=evtdf[('slc','wgt','','','','')].values
        )
        cv_events = total_reco_cv * scale_factor

    return univ_events, cv_events

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def plot_univ_hists(
                univ_events, 
                cv_events,
                syst_name, 
                var_config, 
                approval="internal",
                plot=True,
                save_fig=False, 
                save_name=None,
                use_bin_width = False,
                fig_ext=".pdf",
                dpi=300): 

    assert univ_events.shape[1] == len(cv_events) 
    n_univ = univ_events.shape[0]
    
    # --- 0. Prepare 2D Bin Widths and Labels ---
    # In 2D, "Bin Width" is the Area (dx * dy)
    if use_bin_width:
        dx = np.diff(var_config.bins_x)
        dy = np.diff(var_config.bins_y)
        # Create a meshgrid and flatten it to match the 1D bincount order
        DX, DY = np.meshgrid(dx, dy)
        bin_areas = (DX * DY).flatten() 
        y_label = var_config.xsec_label # Use the label from config
    else:
        bin_areas = 1.0
        y_label = "Events / Bin"

    # Define 1D coordinates for the flattened bins
    flat_bins = np.arange(var_config.n_bins_total + 1)
    flat_bin_centers = np.arange(var_config.n_bins_total) + 0.5
    bin_labels = var_config.get_bin_labels()

    fig, ax = plt.subplots(figsize=(12, 6)) # Wider for many 2D labels

    # Scaling
    plot_cv = cv_events / bin_areas
    plot_univ = univ_events / bin_areas

    # --- 1. Calculate Dynamic Y-Limit ---
    global_max = np.nanmax([np.nanmax(plot_cv), np.nanmax(plot_univ)])
    if global_max <= 0: global_max = 1.0 
    ax.set_ylim(0, global_max * 1.3) # Extra room for legend

    # --- 2. Plot Universes ---
    if (n_univ > 10):
        colors = ["#FDE725FF", "#1F968BFF", "#440154FF"] 
        sorted_univs = np.sort(plot_univ, axis=0)

        # Quantile indices
        idx68 = int(0.68 * n_univ)
        s68, e68 = (n_univ - idx68) // 2, (n_univ - idx68) // 2 + idx68
        idx95 = int(0.95 * n_univ)
        s95, e95 = (n_univ - idx95) // 2, (n_univ - idx95) // 2 + idx95

        segs = [
            (range(s68, e68), colors[0], "Universe (68%)", False),
            (range(s95, e95), colors[1], "Universe (95%)", True),
            ((i for i in range(n_univ) if i not in range(s95, e95)), colors[2], "Universe (100%)", False)
        ]

        plotted = set()
        for r, color, label, skip_inner in segs:
            for i in r:
                if skip_inner and i in range(s68, e68): continue
                show_label = label if label not in plotted else None
                ax.stairs(sorted_univs[i], flat_bins, color=color, alpha=0.4, label=show_label)
                plotted.add(label)
    else:
        for i in range(n_univ):
            show_label = "Universe" if i == 0 else None
            ax.stairs(plot_univ[i], flat_bins, color="gray", alpha=0.5, label=show_label)

    # --- 3. Plot CV last ---
    ax.stairs(plot_cv, flat_bins, color="k", linewidth=2.5, label="Central Value")

    # --- 4. Formatting ---
    ax.set_xlim(flat_bins[0], flat_bins[-1])
    
    # Apply the complex 2D labels to the 1D axis
    ax.set_xticks(flat_bin_centers)
    ax.set_xticklabels(bin_labels, rotation=90, fontsize=8)
    
    ax.set_xlabel("Analysis Bin (Muon $P$ vs $\cos\\theta$)") 
    ax.set_ylabel(y_label)
    ax.set_title(f"{syst_name} - {var_config.var_save_name}", fontsize=14)
    ax.legend(frameon=True, loc='upper right', fontsize=10, ncol=2)
    ax.grid(True, axis='y', linestyle='--', alpha=0.4)

    # Optional: Draw vertical lines to separate the cos_theta blocks
    for i in range(1, var_config.n_bins_y):
        ax.axvline(i * var_config.n_bins_x, color='black', linestyle='-', alpha=0.2)

    # Assumes add_approval_text is defined in your environment
    if 'add_approval_text' in globals():
        add_approval_text(approval, 0.02, 0.98, "left")

    if save_fig and save_name:
        plt.savefig(save_name + fig_ext, bbox_inches='tight', dpi=dpi)

    if plot:
        plt.show()
    else:
        plt.close(fig)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def plot_frac_unc(frac_unc_named_list,  # Expecting [(unc_array, "name"), ...]
                  var_config, 
                  plot_labels=["", "", ""],
                  approval="internal",
                  plot=True,
                  save_fig=False, 
                  save_name=None,
                  fig_ext=".pdf"):

    # --- 0. Prepare 2D-to-1D Binning Logic ---
    n_bins_total = var_config.n_bins_total
    flat_bins = np.arange(n_bins_total + 1)
    flat_bin_centers = np.arange(n_bins_total) + 0.5
    bin_labels = var_config.get_bin_labels()

    fig, ax = plt.subplots(figsize=(12, 6)) # Wider to accommodate 2D labels
    
    max_val = 0
    for fidx, (frac_unc, label_name) in enumerate(frac_unc_named_list):
        # Color Logic
        color = "C{}".format(fidx)
        if len(frac_unc_named_list) == 1 or label_name == "Total":
            color = "black"
            
        # Plotting in percent [%] using ax.stairs
        # Stairs requires the edges, so we use flat_bins
        ax.stairs(frac_unc * 100, flat_bins, 
                  color=color, linewidth=2.5 if label_name == "Total" else 2, 
                  label=label_name)
        
        # Track max for y-limit scaling
        current_max = np.nanmax(np.nan_to_num(frac_unc * 100, nan=0, posinf=0))
        if current_max > max_val:
            max_val = current_max

    # --- 1. Axis Formatting (The 2D Logic) ---
    ax.set_xlim(flat_bins[0], flat_bins[-1])
    
    # Set Major ticks at bin edges
    ax.set_xticks(flat_bins)
    # Set Minor ticks at centers for labels
    ax.set_xticks(flat_bin_centers, minor=True)
    
    # Hide major labels, show rotated 2D labels on minor ticks
    ax.set_xticklabels([], minor=False)
    ax.set_xticklabels(bin_labels, minor=True, rotation=90, fontsize=8)

    ax.set_ylabel("Fractional Uncertainty [%]", fontsize=14)
    ax.set_xlabel("Analysis Bin (Muon $P$ vs $\cos\\theta$)", fontsize=14)
    
    # Set dynamic Y-limit
    if max_val <= 0: max_val = 10.0 
    ax.set_ylim(0, max_val * 1.3) # 1.3 factor for legend clearance
    
    ax.set_title(plot_labels[2], fontsize=16)
    ax.grid(True, axis='y', linestyle='--', alpha=0.4)
    
    # --- 2. Visual Separation of 2D Blocks ---
    # Draw vertical lines every time we move to a new cos_theta bin
    for i in range(1, var_config.n_bins_y + 1):
        ax.axvline(i * var_config.n_bins_x, color='black', linestyle='-', alpha=0.2, linewidth=1)

    ax.legend(loc='upper right', frameon=True, ncol=2 if len(frac_unc_named_list) > 4 else 1)

    # Assumes add_approval_text is available
    if 'add_approval_text' in globals():
        add_approval_text(approval, 0.02, 0.98, "left")
                
    if save_fig and save_name:
        plt.savefig(f"{save_name}{fig_ext}", bbox_inches='tight')

    if plot:
        plt.show()
    else:
        plt.close(fig)

In [ ]:
syst_name = "MCstat"
show_plots = True

syst_dict = {}
for var_config in var_configs:

    univ_events, cv_events = get_univ_rates_2d(cov_type = "rate", evtdf=mc_evt_df,
                                        var_config=VariableConfig2D.muon_momentum_muon_direction(),
                                        n_univ=100,
                                        bkgd_subtract=True,
                                        syst_name=syst_name)
    
    ret_MCstat = get_covariance_matrix(univ_events, cv_events)
    syst_dict[var_config.var_save_name] = ret_MCstat["cov_frac"]

    if show_plots: 
        plot_univ_hists(univ_events, cv_events, syst_name, var_config, use_bin_width = True)

        ret_MCstat = get_covariance_matrix(univ_events, cv_events)
        
        matrix_type = "cov"
        #save_fig_name = "{}/{}-{}-{}.pdf".format(save_fig_dir, var_config.var_save_name, syst_name, matrix_type)
        title = "{} {}".format(syst_name, matrix_type)
        plot_heatmap(ret_MCstat[matrix_type], 
                    bins_flat, 
                    plot_labels=["Bins", "Bins", "Covariance"],
                    save_fig=save_fig, save_name=save_fig_name, tick_labels = labels)
        
        matrix_type = "cov_frac"
        save_fig_name = "{}/{}-{}-{}.pdf".format(save_fig_dir, var_config.var_save_name, syst_name, matrix_type)
        title = "{} {}".format(syst_name, matrix_type)
        plot_heatmap(ret_MCstat[matrix_type], 
                     bins_flat, 
                     plot_labels=["Bins", "Bins", "FractionalCovariance"],
                     save_fig=save_fig, save_name=save_fig_name, tick_labels = labels)
        
        frac_unc = (np.sqrt(np.diag(ret_MCstat["cov_frac"])), "MC Stats")
        plot_frac_unc([frac_unc], var_config)
     
   
# save syst_dict as an npz file in the directory where dfs were loaded from
if save_result:
    print("saving syst_dict as npz in %s" % (file_dir))
    np.savez(file_dir + "/mcstat_syst_dict.npz", **syst_dict)


# Flux

In [ ]:
syst_name = "Flux"

show_plots = True
syst_dict_flux = {}
for var_config in var_configs:
    cov_type = "rate"
    univ_events, cv_events = get_univ_rates_2d(cov_type = cov_type, evtdf=mc_evt_df,
                                    var_config=VariableConfig2D.muon_momentum_muon_direction(),
                                    n_univ=100,
                                    bkgd_subtract=True,
                                    syst_name=syst_name)
    ret_flux = get_covariance_matrix(univ_events, cv_events)
    syst_dict_flux[var_config.var_save_name] = ret_flux["cov_frac"]

    if show_plots: 
        plot_univ_hists(univ_events, cv_events, syst_name, var_config, use_bin_width = True)
        
        frac_unc = (np.sqrt(np.diag(ret_flux["cov_frac"])), "Flux")
        plot_frac_unc([frac_unc], var_config)
    
        matrix_type = "cov"
        save_fig_name = "{}/{}-{}-{}.pdf".format(save_fig_dir, var_config.var_save_name, syst_name, matrix_type)
        title = "{} {}".format(syst_name, matrix_type)
        plot_heatmap(ret_MCstat[matrix_type], 
                 bins_flat, 
                 plot_labels=["Bins", "Bins", "FractionalCovariance"],
                 save_fig=save_fig, save_name=save_fig_name, tick_labels = labels)
        
        matrix_type = "cov_frac"
        save_fig_name = "{}/{}-{}-{}.pdf".format(save_fig_dir, var_config.var_save_name, syst_name, matrix_type)
        title = "{} {}".format(syst_name, matrix_type)
        plot_heatmap(ret_MCstat[matrix_type], 
                     bins_flat, 
                     plot_labels=["Bins", "Bins", "FractionalCovariance"],
                     save_fig=save_fig, save_name=save_fig_name, tick_labels = labels)
        
        matrix_type = "corr"
        save_fig_name = "{}/{}-{}-{}.pdf".format(save_fig_dir, var_config.var_save_name, syst_name, matrix_type)
        title = "{} {}".format(syst_name, matrix_type)
        plot_heatmap(ret_MCstat[matrix_type], 
                     bins_flat, 
                     plot_labels=["Bins", "Bins", "FractionalCovariance"],
                     save_fig=save_fig, save_name=save_fig_name, tick_labels = labels)
    
# save syst_dict as an npz file in the directory where dfs were loaded from
if save_result:
    np.savez(file_dir + "/syst_dict_flux.npz", **syst_dict_flux_rate)

# G4

In [ ]:
syst_name = "G4"

show_plots = True
syst_dict_g4 = {}
for var_config in var_configs:
    cov_type = "rate"
    univ_events, cv_events = get_univ_rates_2d(cov_type = cov_type, evtdf=mc_evt_df,
                                    var_config=VariableConfig2D.muon_momentum_muon_direction(),
                                    n_univ=100,
                                    bkgd_subtract=True,
                                    syst_name=syst_name)
    ret_G4_rate = get_covariance_matrix(univ_events, cv_events)
    syst_dict_g4[var_config.var_save_name] = ret_G4_rate["cov_frac"]

    if show_plots: 
        plot_univ_hists(univ_events, cv_events, syst_name, var_config, use_bin_width = True)
        
        frac_unc = (np.sqrt(np.diag(ret_flux["cov_frac"])), "G4")
        plot_frac_unc([frac_unc], var_config)
    
        matrix_type = "cov"
        save_fig_name = "{}/{}-{}-{}.pdf".format(save_fig_dir, var_config.var_save_name, syst_name, matrix_type)
        title = "{} {}".format(syst_name, matrix_type)
        plot_heatmap(ret_MCstat[matrix_type], 
                 bins_flat, 
                 plot_labels=["Bins", "Bins", "FractionalCovariance"],
                 save_fig=save_fig, save_name=save_fig_name, tick_labels = labels)
        
        matrix_type = "cov_frac"
        save_fig_name = "{}/{}-{}-{}.pdf".format(save_fig_dir, var_config.var_save_name, syst_name, matrix_type)
        title = "{} {}".format(syst_name, matrix_type)
        plot_heatmap(ret_MCstat[matrix_type], 
                     bins_flat, 
                     plot_labels=["Bins", "Bins", "FractionalCovariance"],
                     save_fig=save_fig, save_name=save_fig_name, tick_labels = labels)
        
        matrix_type = "corr"
        save_fig_name = "{}/{}-{}-{}.pdf".format(save_fig_dir, var_config.var_save_name, syst_name, matrix_type)
        title = "{} {}".format(syst_name, matrix_type)
        plot_heatmap(ret_MCstat[matrix_type], 
                     bins_flat, 
                     plot_labels=["Bins", "Bins", "FractionalCovariance"],
                     save_fig=save_fig, save_name=save_fig_name, tick_labels = labels)
    # save syst_dict as an npz file in the directory where dfs were loaded from
if save_result:
    print("saving syst_dict as npz in %s" % (file_dir))
    np.savez(file_dir + "/g4_syst_dict.npz", **syst_dict_g4)

# GENIE

In [ ]:
syst_name = "GENIE"
show_plots = True

syst_dict_rate = {}
syst_dict_xsec = {}

for var_config in var_configs:
    # --- GENIE X-SEC ---
    cov_type = "xsec"
    univ_events, cv_events = get_univ_rates_2d(
        cov_type=cov_type, 
        evtdf=mc_evt_df,
        nudf=mc_nu_df, 
        var_config=var_config, # Fixed: use loop variable
        n_univ=100,
        bkgd_subtract=True,
        syst_name=syst_name
    )
    ret_genie_xsec = get_covariance_matrix(univ_events, cv_events)
    syst_dict_xsec[var_config.var_save_name] = ret_genie_xsec["cov_frac"]

    # --- GENIE RATE ---
    cov_type = "rate"
    univ_events, cv_events = get_univ_rates_2d(
        cov_type=cov_type, 
        evtdf=mc_evt_df,
        var_config=var_config, # Fixed: use loop variable
        n_univ=100,
        bkgd_subtract=True,
        syst_name=syst_name
    )
    ret_genie_rate = get_covariance_matrix(univ_events, cv_events)
    syst_dict_rate[var_config.var_save_name] = ret_genie_rate["cov_frac"]

    if show_plots:
        # Plot Histograms
        plot_univ_hists(univ_events, cv_events, syst_name, var_config, use_bin_width=True)
        
        # Define labels for heatmaps
        labels = var_config.get_bin_labels()
        bins_flat = np.arange(var_config.n_bins_total + 1)

        # Plot Heatmaps for different matrix types
        for matrix_type, label in [("cov", "Covariance"), ("cov_frac", "FractionalCovariance"), ("corr", "Corr")]:
            
            # X-Sec Heatmap
            save_fig_name = f"{save_fig_dir}/{var_config.var_save_name}-{syst_name}_{matrix_type}_xsec.pdf"
            plot_heatmap(ret_genie_xsec[matrix_type], bins_flat, 
                         plot_labels=["Bins", "Bins", label],
                         save_fig=save_fig, save_name=save_fig_name, tick_labels=labels)
            
            # Rate Heatmap
            save_fig_name = f"{save_fig_dir}/{var_config.var_save_name}-{syst_name}_{matrix_type}_rate.pdf"
            plot_heatmap(ret_genie_rate[matrix_type], bins_flat, 
                         plot_labels=["Bins", "Bins", label],
                         save_fig=save_fig, save_name=save_fig_name, tick_labels=labels)

        # --- FIX: Named list for Uncertainty Plot ---
        # This part must be indented inside the 'if show_plots' block
        frac_unc_named_list = [
            (np.sqrt(np.diag(ret_genie_xsec["cov_frac"])), "GENIE x-sec"),
            (np.sqrt(np.diag(ret_genie_rate["cov_frac"])), "GENIE rate")
        ]
        
        plot_frac_unc(frac_unc_named_list, var_config)

# Save results
if save_result:
    print(f"saving syst_dict as npz in {file_dir}")
    np.savez(f"{file_dir}/genie_syst_dict_xsec.npz", **syst_dict_xsec)
    np.savez(f"{file_dir}/genie_syst_dict_rate.npz", **syst_dict_rate)

# All Uncertanties

In [ ]:
Total_Covariance_Frac = ret_flux["cov_frac"] + ret_genie_xsec["cov_frac"] + ret_MCstat["cov_frac"] + ret_G4_rate["cov_frac"]

frac_unc_named_list = [
    (np.sqrt(np.diag(Total_Covariance_Frac)), "Total"),
    (np.sqrt(np.diag(ret_flux["cov_frac"])), "Flux"),
    (np.sqrt(np.diag(ret_genie_xsec["cov_frac"])), "GENIE"),
    (np.sqrt(np.diag(ret_MCstat["cov_frac"])), "MC Stat"),
    (np.sqrt(np.diag(ret_G4_rate["cov_frac"])), "G4"),
]

plot_frac_unc(frac_unc_named_list, var_config)